# cascade: oil sensitivity statistics

This notebook measures how weekly oil-price moves (Brent) show up in the weekly returns of the tickers in the universe (14 core tickers plus 21 extra ones), and how that reaches the three demo portfolios. All the work happens in `compute/stats.py`; each cell below calls one function.

It runs in SageMaker AI and uses the notebook's IAM role to read `s3://$BUCKET/raw/` and write `s3://$BUCKET/processed/`. Set the environment variable `BUCKET` first. Simulation only, not advice.

In [1]:
%env BUCKET=cascade-s3-bucket

env: BUCKET=cascade-s3-bucket


In [2]:
%pip install -r ../requirements.txt
%pip install -U "pyarrow>=15" "pandas>=2.2"


[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # repo root, so `compute` is importable
from compute import stats

BUCKET = os.environ["BUCKET"]
BUCKET

'cascade-s3-bucket'

## Step 1. Load the raw files

Reads the price files (the starter file plus the extra ETFs), Brent oil prices, the USD/CAD exchange rate, the four government bond yields and the 10-year breakeven inflation rate from S3. Where a folder has dated snapshots, it picks the newest one.

In [4]:
raw = stats.load_raw(BUCKET)
{name: df.shape for name, df in raw.items()}

{'prices_starter': (40049, 9),
 'prices_extra': (7530, 9),
 'fx': (1245, 3),
 'macro': (8281, 7),
 'brent': (1304, 2),
 't10yie': (1306, 2)}

## Step 2. Build weekly returns

Keeps the tickers in the universe and turns daily data into weekly data, using the last available value each Friday. Prices, Brent and USD/CAD become weekly log returns. Bond yields and breakeven inflation become weekly changes in percentage points. US-dollar tickers are converted to Canadian dollars by adding the weekly change in USD/CAD. Weeks without a Brent return are dropped. A ticker with fewer than 150 weekly returns is skipped and a warning is printed.

In [5]:
data = stats.build_weekly(raw)
print(data.ret_cad.shape, data.ret_cad.index.min().date(), "to", data.ret_cad.index.max().date())
data.ret_cad.tail(3)

(261, 35) 2021-11-19 to 2026-11-13


ticker,XIU.TO,ENB.TO,XOM,TLT,XBB.TO,ZST.TO,QQQ,SHOP.TO,SPY,VFV.TO,...,TSLA,JPM,JNJ,PG,GLD,AGG,ESGV,XRB.TO,XLB.TO,XEG.TO
observation_date,,,,,,,,,,,,,,,,,,,,,
2026-10-30,-0.002229,0.025500,0.019450,0.003767,0.002618,0.000204,-0.022306,-0.069540,-0.007969,-0.008112,...,0.069143,0.013971,0.032614,0.001049,-0.032347,0.001799,-0.012481,0.002594,0.000669,0.003496
2026-11-06,0.000929,-0.043926,-0.110340,0.016077,0.010494,0.001019,-0.040188,-0.045965,-0.016506,-0.015775,...,-0.098942,0.003306,0.021894,0.034328,0.056878,0.016426,-0.012270,0.012870,0.017768,-0.134257
2026-11-13,0.013287,0.036081,0.033311,-0.017211,-0.009040,-0.000204,0.057734,-0.005049,0.039268,0.037993,...,0.113933,0.034610,0.036096,0.025787,-0.038471,-0.003860,0.037392,-0.020237,-0.013297,0.031080


## Step 3. Sensitivities

For each ticker, estimates how much its weekly CAD return moves when Brent moves: a simple version, a version that also controls for the overall market, a version that leaves out 2022, separate betas for up and down weeks, and a test of how often the sign is right on the biggest oil moves after 2024. Ranges are 95% ranges and use standard errors that allow for correlation over time. Because many tickers are tested at once, `q_simple` adjusts the p-values for multiple testing (Benjamini-Hochberg); `p_simple` itself is unchanged.

In [6]:
sens = stats.sensitivities(data)
sens

,ticker,beta_simple,lo_simple,hi_simple,p_simple,beta_mkt,lo_mkt,hi_mkt,p_mkt,market_ticker,...,lo_ex2022,hi_ex2022,p_ex2022,beta_up,beta_down,oos_hit_rate,oos_n,n_weeks,q_simple,universe
0,XIU.TO,0.010571,-0.032437,0.053579,6.299906e-01,0.010571,-0.032437,0.053579,6.299906e-01,none,...,-0.057657,0.035834,6.473019e-01,-0.065858,0.022323,0.1,10,261,7.112797e-01,core
1,ENB.TO,0.091709,0.038233,0.145184,7.756634e-04,0.083986,0.037310,0.130661,4.207293e-04,XIU.TO,...,0.003522,0.121375,3.778915e-02,0.030145,0.184265,0.6,10,261,3.016469e-03,core
2,XOM,0.357413,0.257646,0.457181,2.193323e-12,0.360900,0.262758,0.459042,5.696957e-13,SPY,...,0.211054,0.423267,4.667726e-09,0.219849,0.408794,0.9,10,261,2.558877e-11,core
3,TLT,-0.089257,-0.125971,-0.052543,1.887754e-06,-0.088706,-0.125824,-0.051587,2.812585e-06,SPY,...,-0.130629,-0.061934,3.924119e-08,-0.033478,-0.069893,0.9,10,261,1.321428e-05,core
4,XBB.TO,-0.034357,-0.051523,-0.017191,8.750171e-05,-0.035779,-0.052312,-0.019246,2.219456e-05,XIU.TO,...,-0.053226,-0.028103,2.222812e-10,-0.016236,-0.014862,1.0,10,261,3.828200e-04,core
5,ZST.TO,-0.000859,-0.001888,0.000169,1.015081e-01,-0.000944,-0.001967,0.000079,7.038517e-02,XIU.TO,...,-0.002101,-0.000121,2.786732e-02,-0.001938,0.001434,0.9,10,261,2.220490e-01,core
6,QQQ,-0.032321,-0.088493,0.023852,2.594253e-01,-0.010715,-0.038556,0.017127,4.506724e-01,SPY,...,-0.094970,0.044269,4.754202e-01,-0.083993,-0.024999,0.3,10,261,4.539943e-01,core
7,SHOP.TO,-0.164236,-0.337716,0.009244,6.351668e-02,-0.192694,-0.400081,0.014693,6.858547e-02,XIU.TO,...,-0.336222,0.048628,1.430055e-01,-0.173361,-0.153149,0.7,10,261,1.587917e-01,core
8,SPY,-0.016677,-0.055082,0.021728,3.947179e-01,-0.016677,-0.055082,0.021728,3.947179e-01,none,...,-0.065304,0.027989,4.330626e-01,-0.066039,0.008192,0.3,10,261,5.526050e-01,core
9,VFV.TO,-0.019721,-0.057846,0.018404,3.106455e-01,-0.019721,-0.057846,0.018404,3.106455e-01,none,...,-0.066793,0.026704,4.006857e-01,-0.064083,0.005276,0.3,10,261,4.978430e-01,core


## Step 4. Rolling beta

Re-estimates the simple beta over a sliding 52-week window for every ticker in the universe, to show whether the relationship is stable or drifts over time.

In [7]:
rolling = stats.rolling_betas(data)
rolling.tail()

,ticker,week,beta_52w,universe
7345,XEG.TO,2026-10-16,0.243083,extra
7346,XEG.TO,2026-10-23,0.238945,extra
7347,XEG.TO,2026-10-30,0.236840,extra
7348,XEG.TO,2026-11-06,0.280730,extra
7349,XEG.TO,2026-11-13,0.275741,extra


## Step 5. Links in the transmission chains

Measures each step of the indirect routes: oil to bond yields, oil to the exchange rate, oil to breakeven inflation, and then yields or the exchange rate to the bond ETFs and equity ETFs they are expected to drive.

In [8]:
links = stats.links(data)
links

,link_id,from_node,to_node,beta,lo,hi,p,unit
0,oil>us10y,oil,us10y,0.489219,0.235419,0.743020,1.580652e-04,pp per 1.0 log
1,oil>ca10y,oil,ca10y,0.514578,0.245078,0.784077,1.822775e-04,pp per 1.0 log
2,oil>usdcad,oil,usdcad,-0.018090,-0.032305,-0.003874,1.262532e-02,log per 1.0 log
3,oil>breakeven,oil,breakeven,0.417310,0.295986,0.538634,1.565609e-11,pp per 1.0 log
4,us10y>TLT,us10y,TLT,-0.135932,-0.148503,-0.123362,1.069637e-99,log per pp
5,ca10y>XBB.TO,ca10y,XBB.TO,-0.065772,-0.068132,-0.063412,0.000000e+00,log per pp
6,ca10y>ZST.TO,ca10y,ZST.TO,-0.001880,-0.002540,-0.001219,2.421699e-08,log per pp
7,usdcad>VFV.TO,usdcad,VFV.TO,-0.188738,-0.583960,0.206485,3.492763e-01,log per 1.0 log
8,us10y>QQQ,us10y,QQQ,-0.014206,-0.050423,0.022010,4.419929e-01,log per pp
9,us10y>SHOP.TO,us10y,SHOP.TO,-0.088918,-0.187049,0.009213,7.573524e-02,log per pp


## Step 6. Chain reconciliation

Multiplies the link betas along each chain (for example oil to US 10-year yield to TLT) and compares the result with the direct oil beta. The share explained shows how much of the direct effect the chain accounts for.

In [9]:
chains = stats.chain_reconciliation(links, sens)
chains

,ticker,chain,beta_direct,beta_chain,share_explained
0,TLT,oil>us10y>TLT,-0.089257,-0.066501,0.745048
1,XBB.TO,oil>ca10y>XBB.TO,-0.034357,-0.033845,0.985093
2,ZST.TO,oil>ca10y>ZST.TO,-0.000859,-0.000967,1.125551
3,VFV.TO,oil>usdcad>VFV.TO,-0.019721,0.003414,-0.173124
4,QQQ,oil>us10y>QQQ,-0.032321,-0.006950,0.215035
5,SHOP.TO,oil>us10y>SHOP.TO,-0.164236,-0.043500,0.264865
6,XLB.TO,oil>ca10y>XLB.TO,-0.063887,-0.063225,0.989637
7,XRB.TO,oil>ca10y>XRB.TO,-0.035254,-0.048473,1.374970


## Step 7. Duration check

For a 10% rise in Brent, works out how far yields should move, then what bond duration implies for the ETF price (TLT, XBB.TO, ZST.TO), and compares it with the move actually measured. Durations are constants at the top of `stats.py`.

In [10]:
duration = stats.duration_check(links, sens)
duration

,ticker,duration_years,yield_move_per_10pct_oil_bp,implied_move_pct,measured_move_pct
0,TLT,14.63,4.662759,-0.682162,-0.850710
1,XBB.TO,6.72,4.904449,-0.329579,-0.327455
2,ZST.TO,0.40,4.904449,-0.019618,-0.008190


## Step 8. Persona holdings and portfolio exposure

Loads the holdings of the three demo investors (robert, amira, mateo) and multiplies each weight by that holding's oil beta. Separately, it builds each investor's weekly portfolio return (weights times the holdings' weekly CAD returns) and regresses that on Brent, which gives the portfolio's own oil beta, range and p-value (`portfolio_beta`).

In [11]:
holdings = stats.persona_holdings()
exposure = stats.portfolio_exposure(holdings, sens)
pbeta = stats.portfolio_beta(data, holdings)
pbeta

,persona,beta,lo,hi,p,r2,n_weeks
0,robert,-0.029800,-0.051227,-0.008374,0.006411,0.025511,261
1,amira,-0.054577,-0.121339,0.012185,0.109098,0.009040,261
2,mateo,-0.004575,-0.041957,0.032807,0.810416,0.000269,261


## Step 9. Scenarios

Applies oil shocks of -30%, -15%, +15% and +30% to every persona and every ticker, and shows the estimated price impact with a low and high end taken from the beta range. Persona rows use the portfolio beta and range from step 8.

In [12]:
scenarios = stats.scenarios(sens, pbeta)
scenarios[scenarios["subject_type"] == "persona"]

,subject,subject_type,shock_pct,impact_pct,lo_pct,hi_pct
0,robert,persona,-30,1.068569,0.299114,1.843926
1,robert,persona,-15,0.485485,0.136181,0.836008
2,robert,persona,15,-0.415628,-0.713399,-0.116964
3,robert,persona,30,-0.778803,-1.335017,-0.219454
4,amira,persona,-30,1.965684,-0.433680,4.422867
5,amira,persona,-15,0.890920,-0.197840,1.991557
6,amira,persona,15,-0.759873,-1.681556,0.170451
7,amira,persona,30,-1.421694,-3.133356,0.320213
8,mateo,persona,-30,0.163321,-1.163309,1.507758
9,mateo,persona,-15,0.074384,-0.531750,0.684212


## Step 10. Alternatives

For each swap in the data contract (for example robert's TLT for ZST.TO), recalculates the portfolio beta after the swap and compares the +30% oil impact before and after.

In [13]:
alternatives = stats.alternatives(holdings, sens)
alternatives

,persona,holding,alternative,beta_holding,beta_alternative,impact30_before_pct,impact30_after_pct
0,robert,TLT,ZST.TO,-0.089257,-0.000859,-0.778803,-0.086043
1,robert,XBB.TO,ZST.TO,-0.034357,-0.000859,-0.778803,-0.429385
2,mateo,XIU.TO,XESG.TO,0.010571,0.014381,-0.119966,-0.070031
3,mateo,VFV.TO,XSP.TO,-0.019721,-0.000250,-0.119966,0.135485
4,amira,QQQ,XIU.TO,-0.032321,0.010571,-1.421694,-1.143974


## Step 11. Risk share, stress history and materiality

Treats each persona and each alternative (after its swap) as one portfolio. Writes the weekly series used everywhere (`weekly_returns`), the share of weekly variance explained by oil, rates and stocks, each on its own (`risk_share`; the three r2 values overlap and do not add up), what the portfolio returned in past large oil moves (`stress_history`, `stress_contributions`; descriptive only, no p-values), and a materiality verdict (`materiality`) with the same rule at 5% and 15% oil r2.

In [ ]:
risk_tables = stats.build_risk_tables(data)
risk_tables["materiality"][["subject", "verdict", "verdict_at_5", "verdict_at_15", "reason_text"]]

## Step 12. Sanity checks and write to S3

First checks results that must hold before anything is published: Exxon (XOM) reacts positively and significantly to oil, the long-bond ETF (TLT) reacts negatively, and every persona has a portfolio beta row. Then writes every table as Parquet to `processed/<table>/part-0.parquet`, plus `processed/_summary/summary.md` with row counts, portfolio betas and the risk summary (it fails if a persona has no rows in `risk_share`, `stress_history`, `stress_contributions` or `materiality`). Existing files are overwritten.

In [14]:
s = sens.set_index("ticker")
assert s.loc["XOM", "beta_simple"] > 0 and s.loc["XOM", "p_simple"] < 0.05, "XOM check failed"
assert s.loc["TLT", "beta_simple"] < 0, "TLT check failed"
assert set(holdings["persona"]) <= set(pbeta["persona"]), "a persona has no portfolio_beta row"

tables = {
    "sensitivities": sens,
    "rolling": rolling,
    "links": links,
    "chain_reconciliation": chains,
    "duration_check": duration,
    "persona_holdings": holdings,
    "portfolio_exposure": exposure,
    "portfolio_beta": pbeta,
    "scenarios": scenarios,
    "alternatives": alternatives,
    **risk_tables,
}
stats.write_outputs(tables, BUCKET)

['processed/sensitivities/part-0.parquet',
 'processed/rolling/part-0.parquet',
 'processed/links/part-0.parquet',
 'processed/chain_reconciliation/part-0.parquet',
 'processed/duration_check/part-0.parquet',
 'processed/persona_holdings/part-0.parquet',
 'processed/portfolio_exposure/part-0.parquet',
 'processed/portfolio_beta/part-0.parquet',
 'processed/scenarios/part-0.parquet',
 'processed/alternatives/part-0.parquet',
 'processed/_summary/summary.md']

In [15]:
import pandas as pd
s3 = "s3://cascade-s3-bucket/processed"
sens = pd.read_parquet(f"{s3}/sensitivities/")
cols = ["ticker","beta_simple","lo_simple","hi_simple","p_simple","beta_ex2022","p_ex2022","oos_hit_rate","n_weeks"]
print(sens[cols].round(3).sort_values("beta_simple").to_string(index=False))
print(pd.read_parquet(f"{s3}/chain_reconciliation/").round(3).to_string(index=False))
print(pd.read_parquet(f"{s3}/duration_check/").round(3).to_string(index=False))
print(open("/dev/stdin") if False else "")
import boto3; print(boto3.client("s3").get_object(Bucket="cascade-s3-bucket", Key="processed/_summary/summary.md")["Body"].read().decode())

 ticker  beta_simple  lo_simple  hi_simple  p_simple  beta_ex2022  p_ex2022  oos_hit_rate  n_weeks
SHOP.TO       -0.164     -0.338      0.009     0.064       -0.144     0.143           0.7      261
     PG       -0.110     -0.164     -0.056     0.000       -0.106     0.001           0.7      261
    TLT       -0.089     -0.126     -0.053     0.000       -0.096     0.000           0.9      261
 CSU.TO       -0.087     -0.216      0.043     0.190       -0.092     0.266           0.6      261
 ATD.TO       -0.074     -0.137     -0.011     0.021       -0.071     0.056           0.3      261
   MSFT       -0.074     -0.153      0.006     0.069       -0.073     0.139           0.5      261
 XLB.TO       -0.064     -0.095     -0.033     0.000       -0.072     0.000           1.0      261
   AMZN       -0.058     -0.138      0.022     0.155       -0.051     0.249           0.7      261
    JNJ       -0.056     -0.113      0.001     0.053       -0.053     0.121           0.7      261
 FTS.TO   